<a href="https://bio-pro.mintlify.app/tools/guides/device-management"><img align="right" src="https://img.shields.io/badge/View_in_Proto_Docs_%E2%86%92-046e7a?style=for-the-badge&logo=readthedocs&logoColor=white" alt="View in Proto Docs: Device Management"></a>

# 03 — Device Management

> **Local inference only.** This notebook covers how `proto_tools` manages models on your own hardware. If you're running via **cloud inference**, this is all handled for you — skip ahead to the next feature.

> Prerequisites: [02 — Tool Persistence](02_tool_persistence.ipynb). Persistent workers are how `DeviceManager` tracks which model is on which GPU.

Tools request devices through the config field `device`. Under the hood, **`DeviceManager`** handles everything:

- **Automatic GPU allocation** — use `ToolInstance.persist()` and `DeviceManager` picks the right device.
- **LRU eviction** — when GPUs are full, the least-recently-used model is offloaded.
- **Multi-GPU packing** — named instances let you run several models in parallel.

**NOTE: You need one or more GPUs to run this notebook.**


In [ ]:
from time import sleep

from proto_tools.tools.structure_prediction.esmfold import (
    ESMFoldConfig,
    ESMFoldInput,
    run_esmfold,
)
from proto_tools.utils import display_gpu_memory_usage
from proto_tools.utils.device_manager import DeviceManager, OffloadStrategy
from proto_tools.utils.tool_instance import ToolInstance

TEST_SEQ = "MKTLLILAVVAAALA"
TEST_SEQ2 = "GAVLTVLLGGLLLA"

---
## 1. Requesting a device

Every tool config has a `device` field. GPU tools default to `"cuda"`, CPU tools default to `"cpu"`. Requests fall into two categories:

### General requests (DeviceManager chooses the GPU)

With a general request, `DeviceManager` finds available GPUs, handles allocation, and triggers LRU eviction if everything is busy. This is the recommended approach — you don't need to know which GPUs are free.

| Value | Meaning |
|---|---|
| `"cpu"` | Run on CPU |
| `"cuda"` | Run on one GPU — `DeviceManager` picks which one |
| `"cudax2"`, `"cudax3"`, … | Run on N GPUs — `DeviceManager` picks which ones (for multi-GPU models) |

### Specific requests (you choose the GPU)

With a specific request, `DeviceManager` places the model on exactly the device(s) you asked for. If that device is already occupied, it evicts whatever is there. Use this when you need precise control.

| Value | Meaning |
|---|---|
| `"cuda:0"` | Run on exactly GPU 0 |
| `"cuda:0,1"` or `"cuda:0,cuda:1"` | Run on exactly GPUs 0 and 1 (for multi-GPU models) |


### Example 1a — General vs specific

With a **general** request (`device="cuda"`), `DeviceManager` picks a free GPU for you. With a **specific** request (`device="cuda:2"`), you choose exactly which GPU the model lands on.

(This example assumes you have at least 3 visible GPUs. If not, change `"cuda:2"` to a GPU ID you have.)


In [ ]:
print("Before:")
display_gpu_memory_usage()

# --- General request: DeviceManager picks the GPU ---
print("\n=== General request: device='cuda' ===")
with ToolInstance.persist():
    # Note: "cuda" is the GPU-tool default, so the config is optional
    output = run_esmfold(ESMFoldInput(complexes=[TEST_SEQ]))
    print("Model loaded:")
    display_gpu_memory_usage()

print("\n(cleaned up)")
display_gpu_memory_usage()

# --- Specific request: you pick the GPU ---
print("\n=== Specific request: device='cuda:2' ===")
with ToolInstance.persist():
    output = run_esmfold(
        ESMFoldInput(complexes=[TEST_SEQ]),
        ESMFoldConfig(device="cuda:2"),
    )
    print("Model loaded:")
    display_gpu_memory_usage()

print("\n(cleaned up)")
display_gpu_memory_usage()

### Example 1b — Limiting the managed pool

By default, `DeviceManager` auto-detects all `CUDA_VISIBLE_DEVICES` and allocates from the full pool. You can restrict it to a subset via `configure(managed_devices=...)` or the `BIO_TOOLS_MANAGED_DEVICES` environment variable.

Useful when you want to reserve some GPUs for other work, or share a large box with other processes.


In [ ]:
# Default: DeviceManager sees all GPUs
dm = DeviceManager.get_instance()
print("Default pool:", dm.get_device_status()["available_devices"])

# Restrict to just cuda:1
DeviceManager.reset_instance()
dm = DeviceManager.get_instance()
dm.configure(managed_devices=["cuda:1"])
print("Restricted pool:", dm.get_device_status()["available_devices"])

# General requests now only land on cuda:1
with ToolInstance.persist():
    output = run_esmfold(ESMFoldInput(complexes=[TEST_SEQ]))
    print("\nModel loaded (general request, restricted pool):")
    display_gpu_memory_usage()

# Equivalent via environment variable (set before the process starts):
#   export BIO_TOOLS_MANAGED_DEVICES="cuda:1"

# Reset back to the full pool for the rest of the notebook
DeviceManager.reset_instance()

---
## 2. Eviction strategies

When every managed GPU is full and a new model needs to load, `DeviceManager` evicts the **least recently used** model. This works across any number of GPUs — the LRU model is picked globally, regardless of which device it's on.

Two strategies control what happens to the evicted model:

- **`RESTART`** (default) — the worker is shut down entirely. Frees all memory, but reloading later pays the full startup cost.
- **`CPU`** — the model is moved to CPU RAM. Stays loaded and can be moved back to GPU quickly, at the cost of system memory.


### Example 2a — RESTART (default)

The evicted model is shut down entirely. GPU memory is fully freed; the next call to the evicted instance pays the full reload.


In [ ]:
# Limit to 1 device so eviction is guaranteed
DeviceManager.reset_instance()
dm = DeviceManager.get_instance()
dm.configure(managed_devices=["cuda:0"])

def show_status():
    """Print allocations and GPU memory."""
    allocs = dm.get_device_status()["allocations"]
    for name, info in allocs.items():
        print(f"  {name}: {info['device_id']}")
    if not allocs:
        print("  (no allocations)")
    display_gpu_memory_usage()

print("=== Load instance A ===")
with ToolInstance.persist_tool("esmfold", instance_name="A"):
    run_esmfold(ESMFoldInput(complexes=[TEST_SEQ]), instance="A")
    show_status()

    print("\n=== Load instance B (RESTART evicts A — fully shut down) ===")
    with ToolInstance.persist_tool("esmfold", instance_name="B"):
        sleep(0.1)
        run_esmfold(ESMFoldInput(complexes=[TEST_SEQ2]), instance="B")
        show_status()

DeviceManager.reset_instance()

### Example 2b — CPU offload

With `OffloadStrategy.CPU`, the evicted model stays in RAM instead of being shut down. Moving it back to GPU later is fast.


In [ ]:
DeviceManager.reset_instance()
dm = DeviceManager.get_instance()
dm.configure(managed_devices=["cuda:0"], offload_strategy=OffloadStrategy.CPU)

def show_status():
    allocs = dm.get_device_status()["allocations"]
    for name, info in allocs.items():
        print(f"  {name}: {info['device_id']}")
    display_gpu_memory_usage()

print("=== Load instance A ===")
with ToolInstance.persist_tool("esmfold", instance_name="A"):
    run_esmfold(ESMFoldInput(complexes=[TEST_SEQ]), instance="A")
    show_status()

    print("\n=== Load instance B (evicts A to CPU) ===")
    with ToolInstance.persist_tool("esmfold", instance_name="B"):
        sleep(0.1)
        run_esmfold(ESMFoldInput(complexes=[TEST_SEQ2]), instance="B")
        show_status()

        print("\n=== Run A again (fast CPU → GPU reload; evicts B to CPU) ===")
        sleep(0.1)
        run_esmfold(ESMFoldInput(complexes=[TEST_SEQ]), instance="A")
        show_status()

DeviceManager.reset_instance()

### Example 2c — LRU across multiple GPUs

With 2 GPUs and 3 instances, the first two fill both GPUs. The third triggers LRU eviction of whichever instance was used least recently.


In [ ]:
DeviceManager.reset_instance()
dm = DeviceManager.get_instance()
dm.configure(managed_devices=["cuda:0", "cuda:1"])

def show_status():
    allocs = dm.get_device_status()["allocations"]
    for name, info in allocs.items():
        print(f"  {name}: {info['device_id']}")
    if not allocs:
        print("  (no allocations)")
    display_gpu_memory_usage()

print("=== Load A (gets cuda:0) ===")
with ToolInstance.persist_tool("esmfold", instance_name="A"):
    run_esmfold(ESMFoldInput(complexes=[TEST_SEQ]), instance="A")
    show_status()

    print("\n=== Load B (gets cuda:1) ===")
    with ToolInstance.persist_tool("esmfold", instance_name="B"):
        sleep(0.1)
        run_esmfold(ESMFoldInput(complexes=[TEST_SEQ2]), instance="B")
        show_status()

        print("\n=== Load C (evicts A — the LRU — from cuda:0) ===")
        with ToolInstance.persist_tool("esmfold", instance_name="C"):
            sleep(0.1)
            run_esmfold(ESMFoldInput(complexes=["MGQQPGKVLGDQRR"]), instance="C")
            show_status()

DeviceManager.reset_instance()

---
## 3. Multiple models per device

By default `DeviceManager` allows **one model per GPU** — loading a second triggers eviction. If you have headroom, set `allow_multiple_per_device=True` and it will pack them together.

**Caveat.** `DeviceManager` doesn't track model sizes or estimate memory usage — if you overcommit you'll get an OOM error from the GPU. Memory-aware packing is non-trivial (other processes leave residue, CLI tools don't expose their footprint, memory scales with input size) so it's on you to make sure the models you pack actually fit.


### Example 3 — Packing 4 instances on 2 GPUs

With `allow_multiple_per_device=True`, instances round-robin across the available GPUs instead of evicting.


In [ ]:
DeviceManager.reset_instance()
dm = DeviceManager.get_instance()
dm.configure(managed_devices=["cuda:0", "cuda:1"], allow_multiple_per_device=True)

def show_status():
    allocs = dm.get_device_status()["allocations"]
    for name, info in allocs.items():
        print(f"  {name}: {info['device_id']}")
    display_gpu_memory_usage()

with ToolInstance.persist_tool("esmfold", instance_name="A"):
    with ToolInstance.persist_tool("esmfold", instance_name="B"):
        with ToolInstance.persist_tool("esmfold", instance_name="C"):
            with ToolInstance.persist_tool("esmfold", instance_name="D"):

                print("=== Load A ===")
                run_esmfold(ESMFoldInput(complexes=[TEST_SEQ]), instance="A")
                show_status()

                print("\n=== Load B ===")
                run_esmfold(ESMFoldInput(complexes=[TEST_SEQ2]), instance="B")
                show_status()

                print("\n=== Load C ===")
                run_esmfold(ESMFoldInput(complexes=["MGQQPGKVLGDQRR"]), instance="C")
                show_status()

                print("\n=== Load D ===")
                run_esmfold(ESMFoldInput(complexes=["ASTVKFLGPVLDAA"]), instance="D")
                show_status()

                print("\n4 instances, 2 GPUs, no evictions!")

DeviceManager.reset_instance()

## What's next?

- **[04 — Parallel Execution](04_parallel_execution.ipynb)** — `ToolPool` builds on `DeviceManager` to automatically fan out a batch of work across every managed GPU.
